# Custom Coupling Topologies

This notebook explores how different coupling topologies affect
oscillator dynamics in PRINet. We'll build custom topologies,
measure synchronization, and study phase-amplitude coupling (PAC).

**Prerequisites:** Install PRINet from source (`pip install -e ".[dev]"`)


In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
import math
from prinet import (
    KuramotoOscillator,
    OscillatorState,
    kuramoto_order_parameter,
    PhaseAmplitudeCoupling,
)
from prinet.utils.oscillosim import (
    OscilloSim,
    ring_topology,
    small_world_topology,
    cosine_coupling_kernel,
    local_order_parameter,
    chimera_index,
    strength_of_incoherence,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 1. Built-in Topologies

PRINet provides several coupling topologies out of the box:

| Topology | Function | Description |
|----------|----------|-------------|
| Ring | `ring_topology(N, k)` | Each oscillator couples to $k$ nearest neighbors on a ring |
| Small-world | `small_world_topology(N, k, p)` | Watts-Strogatz: ring + random rewiring |
| Sparse k-NN | OscilloSim `"sparse_knn"` | Random $k$-nearest-neighbor graph |
| Mean-field | OscilloSim `"mean_field"` | All-to-all (Kuramoto mean field) |

In [ ]:
N = 256
k = 8

# Ring topology
ring = ring_topology(N, k, device=DEVICE)
print(f"Ring: shape={ring.shape}, dtype={ring.dtype}")
print(f"  Oscillator 0 neighbors: {ring[0].cpu().tolist()}")

# Small-world topology
sw = small_world_topology(N, k, p_rewire=0.2, device=DEVICE, seed=42)
print(f"Small-world: shape={sw.shape}")
print(f"  Oscillator 0 neighbors: {sw[0].cpu().tolist()}")

## 2. Building a Custom Topology

Create a two-cluster topology: two groups of oscillators with
dense intra-group coupling and sparse inter-group coupling.

In [ ]:
def two_cluster_topology(
    N: int, k_intra: int, k_inter: int,
    device: str = "cpu", seed: int = 42,
) -> torch.Tensor:
    """Create a two-cluster neighbor index.
    
    Each oscillator has k_intra neighbors within its cluster
    and k_inter neighbors in the other cluster.
    """
    gen = torch.Generator(device=device).manual_seed(seed)
    half = N // 2
    k_total = k_intra + k_inter
    nbr = torch.zeros(N, k_total, dtype=torch.long, device=device)
    
    for i in range(N):
        if i < half:
            own_pool = torch.arange(0, half, device=device)
            other_pool = torch.arange(half, N, device=device)
        else:
            own_pool = torch.arange(half, N, device=device)
            other_pool = torch.arange(0, half, device=device)
        
        # Remove self from own pool
        own_pool = own_pool[own_pool != i]
        
        # Sample neighbors
        intra_idx = torch.randint(0, len(own_pool), (k_intra,),
                                  device=device, generator=gen)
        inter_idx = torch.randint(0, len(other_pool), (k_inter,),
                                  device=device, generator=gen)
        nbr[i, :k_intra] = own_pool[intra_idx]
        nbr[i, k_intra:] = other_pool[inter_idx]
    
    return nbr

# Create topology
N = 200
nbr_custom = two_cluster_topology(N, k_intra=8, k_inter=2, device=DEVICE)
print(f"Custom topology: {nbr_custom.shape}")

# Visualize adjacency structure
adj = torch.zeros(N, N)
for i in range(N):
    for j in nbr_custom[i].cpu():
        adj[i, j.item()] = 1

plt.figure(figsize=(6, 6))
plt.imshow(adj.numpy(), cmap="Blues", interpolation="nearest")
plt.axhline(N // 2 - 0.5, color="red", linewidth=1)
plt.axvline(N // 2 - 0.5, color="red", linewidth=1)
plt.title("Two-cluster adjacency matrix")
plt.xlabel("Oscillator j")
plt.ylabel("Oscillator i")
plt.colorbar(label="Connected")
plt.tight_layout()
plt.show()

## 3. Simulating with Custom Topology

Feed the custom neighbor indices to OscilloSim via `coupling_weights`
or build a simulation manually with `KuramotoOscillator`:

In [ ]:
# Use OscilloSim with sparse_knn mode and custom neighbors
# We'll run two simulations: one with the two-cluster and one mean-field

configs = {
    "Mean-field": dict(coupling_mode="mean_field"),
    "Ring (k=10)": dict(coupling_mode="ring", k_neighbors=10),
    "Small-world (p=0.1)": dict(coupling_mode="small_world", k_neighbors=10, p_rewire=0.1),
    "Small-world (p=0.5)": dict(coupling_mode="small_world", k_neighbors=10, p_rewire=0.5),
}

fig, ax = plt.subplots(figsize=(9, 5))

for name, kwargs in configs.items():
    sim = OscilloSim(
        n_oscillators=N,
        coupling_strength=2.5,
        device=DEVICE,
        seed=42,
        **kwargs,
    )
    res = sim.run(n_steps=500, dt=0.01, record_trajectory=True, record_interval=5)
    ax.plot(res.order_parameter, label=name)

ax.set_xlabel("Recording step")
ax.set_ylabel("Order parameter r")
ax.set_title("Synchronization dynamics by topology")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Phase-Amplitude Coupling (PAC)

PAC modulates fast oscillator amplitudes by slow oscillator phases.
This is how PRINet implements cross-frequency binding.

In [ ]:
pac = PhaseAmplitudeCoupling(modulation_depth=0.5)

# Slow phase: 10 oscillators
slow_phase = torch.linspace(0, 2 * math.pi, 100, device=DEVICE)
# Fast amplitude: 20 oscillators
fast_amp = torch.ones(20, device=DEVICE)

# Sweep slow phase and measure PAC modulation
phi_vals = torch.linspace(0, 2 * math.pi, 200, device=DEVICE)
modulated = []

for phi in phi_vals:
    sp = torch.full((1,), phi.item(), device=DEVICE)
    fa = torch.ones(1, device=DEVICE)
    out = pac.modulate(sp, fa)
    modulated.append(out.item())

plt.figure(figsize=(8, 4))
plt.plot(phi_vals.cpu().numpy(), modulated, linewidth=2)
plt.xlabel("Slow phase (rad)")
plt.ylabel("Modulated amplitude")
plt.title(f"PAC modulation (depth={pac.modulation_depth})")
plt.axhline(1.0, color="gray", linestyle="--", alpha=0.5, label="Unmodulated")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Abrams-Strogatz Cosine Kernel

The cosine coupling kernel implements distance-dependent coupling
weights on a ring, used for chimera state research:

In [ ]:
N_ring = 128
k_ring = 20

weights = cosine_coupling_kernel(N_ring, k_ring, A=0.995, device=DEVICE)

# Visualize the coupling profile for oscillator 0
plt.figure(figsize=(8, 4))
w = weights[0].cpu().numpy()
nbr_ring = ring_topology(N_ring, k_ring, device=DEVICE)
nbr_ring_0 = nbr_ring[0].cpu().numpy()

plt.bar(range(k_ring), w, alpha=0.7)
plt.xlabel("Neighbor index")
plt.ylabel("Coupling weight")
plt.title("Cosine coupling kernel (Abrams-Strogatz)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Rewiring Probability and Synchronization Speed

Watts-Strogatz small-world networks show faster synchronization
with higher rewiring probability $p$:

In [ ]:
p_values = [0.0, 0.01, 0.05, 0.1, 0.3, 0.5, 1.0]
sync_times = []

for p in p_values:
    sim = OscilloSim(
        n_oscillators=500,
        coupling_strength=3.0,
        coupling_mode="small_world",
        k_neighbors=10,
        p_rewire=p,
        device=DEVICE,
        seed=42,
    )
    res = sim.run(n_steps=500, dt=0.01, record_trajectory=True, record_interval=1)
    
    # Find first step where r > 0.9
    sync_step = next(
        (i for i, r in enumerate(res.order_parameter) if r > 0.9),
        len(res.order_parameter),
    )
    sync_times.append(sync_step)

plt.figure(figsize=(7, 4))
plt.plot(p_values, sync_times, "o-", markersize=6)
plt.xlabel("Rewiring probability p")
plt.ylabel("Steps to reach r > 0.9")
plt.title("Small-world effect on synchronization speed")
plt.xscale("log" if min(p_values) > 0 else "linear")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

| Topic | Key API |
|-------|--------|
| Ring coupling | `ring_topology(N, k)` |
| Small-world | `small_world_topology(N, k, p_rewire)` |
| Cosine kernel | `cosine_coupling_kernel(N, k, A)` |
| PAC | `PhaseAmplitudeCoupling(modulation_depth)` |
| Custom topology | Build `(N, k)` neighbor index tensor |

PRINet's modular design makes it easy to experiment with novel
coupling structures. Custom topologies are just `(N, k)` integer
tensors indexing neighbors.